### 馬達論文研究
### 第五步 重新模型建立
### 馬達A 8000RPM
### CNN_11 Layers
### 10種工況

In [ ]:
# --- logging bootstrap (auto-added) ---
from scripts.notebook_bootstrap import bootstrap

LOG, RUN_PATHS = bootstrap()
# --- end logging bootstrap ---


In [ ]:
# 匯入所需函式庫
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
from scripts.gpu_utils import device_scope, DEVICE
warnings.filterwarnings("ignore")

In [ ]:
# 資料準備與處理
def load_and_preprocess_data(base_dir, screws_list):
    datasets = []
    for screws in screws_list:
        file_path = os.path.join(base_dir, screws, 'T3_Group_feature_data.csv')
        if os.path.exists(file_path):
            dataset = pd.read_csv(file_path)
            dataset['screws'] = screws
            datasets.append(dataset)
        else:
            print(f"檔案未找到: {file_path}")
    return datasets

In [ ]:
# 新資料
load_and_preprocess_newdata = load_and_preprocess_data


In [ ]:
# 加載資料
rootDir = os.getcwd()
myfeatureDirectory = os.path.join(rootDir, 'data', 'Step-3', 'myfeature', 'T3', '8000rpm')
screws_list = ['8screws', '1screws', '2screws', '3screws', '4screws']
datasets_T1 = load_and_preprocess_data(myfeatureDirectory, screws_list)

In [ ]:
myfeatureDirectory2 = os.path.join(rootDir, 'data', 'Step-3', 'myfeature', 'T3', '8000rpm')
screws_list2 = ['5screws', '6screws', '7screws', '3_14screws', '4_146screws']
datasets_T2 = load_and_preprocess_newdata(myfeatureDirectory2, screws_list2)

In [ ]:
# 合併資料
combined_data = pd.concat(datasets_T1 + datasets_T2, ignore_index=True)

In [ ]:
# **手動定義標籤順序**
desired_order = ['8screws', '1screws', '2screws', '3screws', '4screws', '5screws', '6screws', '7screws', '3_14screws', '4_146screws']
label_mapping = {screw: idx for idx, screw in enumerate(desired_order)}
combined_data['screws'] = combined_data['screws'].map(label_mapping)

In [ ]:
# 確認標籤映射是否正確
print("自訂 Label Mapping:")
for label, num in label_mapping.items():
    print(f"{label}: {num}")

In [ ]:
# 資料分割
X = combined_data.drop(['screws'], axis=1).values
y_screws = combined_data['screws'].values

In [ ]:
# 劃分訓練與測試集
X_train, X_test, y_train_screws, y_test_screws = train_test_split(
    X, y_screws, test_size=0.2, random_state=42, stratify=y_screws
)

In [ ]:
# 標準化特徵
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# 建立 CNN 模型
def build_cnn_model(input_shape, num_classes):

    # 定義輸入層
    input_layer = tf.keras.Input(shape=input_shape)

    # 第一層卷積層
    x = Conv1D(16, kernel_size=3, activation='relu', padding='same')(input_layer)

    # 第二層卷積層
    x = Conv1D(16, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # 第三層卷積層
    x = Conv1D(16, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # 第四層卷積層
    x = Conv1D(16, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Flatten()(x)

    # 全連接層
    x = Dense(16, activation='relu')(x)
    x = Dropout(0.3)(x)

    # 輸出層
    output = Dense(num_classes, activation='softmax', name="Screw_Number_Output")(x)


    # 定義模型    
    model = Model(inputs=input_layer, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss=["sparse_categorical_crossentropy"],
        metrics=["accuracy"]
    )
    return model

In [ ]:
# 模型參數
num_screw_classes = len(np.unique(y_train_screws))

In [ ]:
print(f"[Training] Device: {DEVICE}")
with device_scope():
    # 建立與訓練模型
    cnn_model = build_cnn_model((X_train.shape[1], 1), num_screw_classes)
    history = cnn_model.fit(
        X_train.reshape(X_train.shape[0], X_train.shape[1], 1),
        y_train_screws,
        validation_data=(
            X_test.reshape(X_test.shape[0], X_test.shape[1], 1),
            y_test_screws
        ),
        epochs=100,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]
    )

In [ ]:
# 模型架構
cnn_model.summary()

In [ ]:
# 評估模型
results = cnn_model.evaluate(
    X_test.reshape(X_test.shape[0], X_test.shape[1], 1),
    y_test_screws
)
print(f"Screw Number Loss: {results[0]:.4f}, Screw Number Accuracy: {results[1]:.4f}")

In [ ]:
# 繪製學習曲線
plt.figure()
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Screw Number Learning Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure()
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Screw Number Learning Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# 混淆矩陣繪製
def plot_confusion_matrix(y_true, y_pred, class_labels, title):
    """
    繪製混淆矩陣，顯示數字映射標籤
    """
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

# 預測並繪製混淆矩陣
y_pred_screw = cnn_model.predict(X_test.reshape(X_test.shape[0], X_test.shape[1], 1)).argmax(axis=1)
plot_confusion_matrix(y_test_screws, y_pred_screw, list(range(len(desired_order))), "Confusion Matrix - Model 19 - New Fault Diagnosis Model")

In [ ]:
# 儲存重訓練模型
modelDirectory = os.path.join(rootDir, 'data', 'Step-3', 'model')
os.makedirs(modelDirectory, exist_ok=True)
modelPath = os.path.join(modelDirectory, 'CNN_A8000_retrained.keras')
cnn_model.save(modelPath)
print(f"重訓練模型已儲存：{modelPath}")